In [1]:
import pandas as pd

In [2]:
from langchain_groq import ChatGroq
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage
from langgraph.prebuilt import create_react_agent
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS
from langchain_community.embeddings import HuggingFaceEmbeddings

In [5]:
from langchain_groq import ChatGroq
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage
from langgraph.prebuilt import create_react_agent
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS
from langchain_community.embeddings import HuggingFaceEmbeddings

GROQ_API_KEY = "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"

llm = ChatGroq(
    model="llama-3.3-70b-versatile",
    temperature=0.5,
    api_key=GROQ_API_KEY
)

# =====================================
# RAG Tool (PDF Reader)
# =====================================

@tool
def search_pdf(query: str) -> str:
    """
    Search information from the uploaded PDF document.
    Use this tool when user asks about the document content.
    """
    try:
        # Load PDF
        loader = PyPDFLoader(r"D:\D\work\code\file-sample_150kB.pdf")   # ← Apna PDF path yahan daalo
        documents = loader.load()

        # Split text
        text_splitter = RecursiveCharacterTextSplitter(chunk_size=700, chunk_overlap=50)
        chunks = text_splitter.split_documents(documents)

        # Create vector store
        embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
        vectorstore = FAISS.from_documents(chunks, embeddings)

        # Search
        results = vectorstore.similarity_search(query, k=4)
        return "\n\n".join([doc.page_content for doc in results])
    
    except Exception as e:
        return f"Error reading PDF: {str(e)}"


# Normal Tools
@tool
def get_current_time() -> str:
    """Returns current date and time"""
    from datetime import datetime
    return datetime.now().strftime("%Y-%m-%d %H:%M:%S")


@tool
def calculator(expression: str) -> str:
    """Perform math calculations"""
    try:
        return str(eval(expression))
    except:
        return "Invalid expression"


tools = [search_pdf, get_current_time, calculator]

# =====================================
# Create Agent
# =====================================

agent = create_react_agent(llm, tools)

print("✅ Advanced RAG Agent Ready!")
print("Ab PDF ke baare mein sawal pooch sakte ho\n")

# =====================================
# Chat Loop
# =====================================

while True:
    user_input = input("You: ").strip()
    
    if user_input.lower() in ["exit", "quit", "bye"]:
        print("👋 Goodbye!")
        break
    
    if not user_input:
        continue

    result = agent.invoke({
        "messages": [HumanMessage(content=user_input)]
    })
    
    print(f"Agent: {result['messages'][-1].content}\n")

✅ Advanced RAG Agent Ready!
Ab PDF ke baare mein sawal pooch sakte ho



C:\Users\Ali Mehdi\AppData\Local\Temp\ipykernel_2568\1018455115.py:72: LangGraphDeprecatedSinceV10: create_react_agent has been moved to `langchain.agents`. Please update your import to `from langchain.agents import create_agent`. Deprecated in LangGraph V1.0 to be removed in V2.0.
  agent = create_react_agent(llm, tools)


BadRequestError: Error code: 400 - {'error': {'message': "Failed to call a function. Please adjust your prompt. See 'failed_generation' for more details.", 'type': 'invalid_request_error', 'code': 'tool_use_failed', 'failed_generation': '<function=search_pdf{"query": "document purpose"}</function>'}}

In [8]:
from langchain_groq import ChatGroq
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage
from langgraph.prebuilt import create_react_agent
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS
from langchain_community.embeddings import HuggingFaceEmbeddings

GROQ_API_KEY = "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"

llm = ChatGroq(
    model="llama-3.3-70b-versatile",
    temperature=0.3,          # Kam temperature better tool calling ke liye
    api_key=GROQ_API_KEY
)

# =====================================
# RAG Tool
# =====================================

@tool
def search_pdf(query: str) -> str:
    """Search and retrieve information from the PDF document."""
    try:
        loader = PyPDFLoader(r"D:\D\work\code\hammad.pdf")   # ← Apna path daalo
        docs = loader.load()

        splitter = RecursiveCharacterTextSplitter(chunk_size=600, chunk_overlap=50)
        chunks = splitter.split_documents(docs)

        embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
        vectorstore = FAISS.from_documents(chunks, embeddings)

        results = vectorstore.similarity_search(query, k=3)
        return "\n\n".join([doc.page_content for doc in results])
    except Exception as e:
        return f"Error: {str(e)}"


@tool
def calculator(expression: str) -> str:
    """Perform math calculations. Example: 25*4"""
    try:
        return str(eval(expression))
    except:
        return "Invalid math expression"


tools = [search_pdf, calculator]

# =====================================
# Agent
# =====================================

agent = create_react_agent(llm, tools)

print("✅ RAG Agent Ready!\n")

while True:
    user_input = input("You: ").strip()
    if user_input.lower() in ["exit", "quit", "bye"]:
        print("👋 Goodbye!")
        break
    if not user_input:
        continue

    result = agent.invoke({"messages": [HumanMessage(content=user_input)]})
    
    # Final answer print
    print(f"Agent: {result['messages'][-1].content}\n")

✅ RAG Agent Ready!



C:\Users\Ali Mehdi\AppData\Local\Temp\ipykernel_2568\1645172890.py:56: LangGraphDeprecatedSinceV10: create_react_agent has been moved to `langchain.agents`. Please update your import to `from langchain.agents import create_agent`. Deprecated in LangGraph V1.0 to be removed in V2.0.
  agent = create_react_agent(llm, tools)


Agent: I couldn't find any math expression on your message, if you need help with something else, please let me know.

Agent: Lorem Ipsum is a placeholder text used to fill space in design and publishing. It is a scrambled version of a Latin text, often used to demonstrate the layout and visual elements of a document or website without the distraction of meaningful content. The text is derived from a Latin text by Cicero, with words and phrases altered to create a nonsensical but visually appealing sequence of letters.

👋 Goodbye!


In [17]:
while True:
    user_input = input("You: ").strip()
    if user_input.lower() in ["exit", "quit", "bye"]:
        print("👋 Goodbye!")
        break
    if not user_input:
        continue

    result = agent.invoke({
        "messages": [HumanMessage(content=user_input)]
    })
    
    print(f"Agent: {result['messages'][-1].content}\n")

ValueError: Checkpointer requires one or more of the following 'configurable' keys: thread_id, checkpoint_ns, checkpoint_id

In [25]:
from langchain_groq import ChatGroq
from langchain_core.messages import HumanMessage, AIMessage, SystemMessage
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS
from langchain_community.embeddings import HuggingFaceEmbeddings

GROQ_API_KEY = "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"
PDF_PATH = r"D:\D\work\code\file-sample_150kB.pdf"

# === PDF Load (ek baar) ===
print("📄 PDF load ho raha hai...")
loader = PyPDFLoader(PDF_PATH)
chunks = RecursiveCharacterTextSplitter(
    chunk_size=700, chunk_overlap=50
).split_documents(loader.load())

vectorstore = FAISS.from_documents(
    chunks,
    HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
)
print("✅ PDF Ready!\n")

# === LLM ===
llm = ChatGroq(
    model="llama-3.3-70b-versatile",  # yeh stable hai
    temperature=0,
    api_key=GROQ_API_KEY
)

# === Memory ===
chat_history = [
    SystemMessage(content="""Tu ek smart assistant hai.
- PDF se context milega toh use karke jawab de
- Pichli baatein yaad rakh
- Clear aur simple jawab de""")
]

def search_pdf(query):
    results = vectorstore.similarity_search(query, k=3)
    return "\n\n".join([d.page_content for d in results]) if results else None

def calculator(expression):
    try:
        allowed = set("0123456789+-*/(). ")
        if not all(c in allowed for c in expression):
            return "Invalid"
        return str(eval(expression))
    except:
        return "Invalid expression"

print("🤖 Agent Ready! (exit likhو band karne ke liye)\n")

while True:
    try:
        user_input = input("You: ").strip()
    except (EOFError, KeyboardInterrupt):
        print("\n👋 Goodbye!")
        break

    if not user_input:
        continue
    if user_input.lower() in ["exit", "quit", "bye"]:
        print("👋 Goodbye!")
        break

    # Calculator check
    if any(op in user_input for op in ["+", "-", "*", "/"]):
        print(f"🤖 Bot: {calculator(user_input)}\n")
        continue

    # PDF context attach karo
    pdf_context = search_pdf(user_input)
    if pdf_context:
        message = f"User: {user_input}\n\nPDF Context:\n{pdf_context}"
    else:
        message = user_input

    chat_history.append(HumanMessage(content=message))

    try:
        response = llm.invoke(chat_history)
        chat_history.append(AIMessage(content=response.content))
        print(f"\n🤖 Bot: {response.content}\n")
    except Exception as e:
        chat_history.pop()
        print(f"❌ Error: {e}\n")

📄 PDF load ho raha hai...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

✅ PDF Ready!

🤖 Agent Ready! (exit likhو band karne ke liye)


🤖 Bot: Lorem ipsum ek jargon hai jo aksar design aur printing me use hota hai. Yah ek placeholder text hai jo asli content ke jagah par use kiya jata hai, jab tak ki asli content available nahi hota. Iska matlab hai ki yah ek temporary text hai jo design aur layout ko dekhne ke liye use kiya jata hai, lekin yah koi meaningful information nahi deta.


🤖 Bot: Aapne pehle "lorem ipsum kia ha" pucha tha. Maine aapko bataya tha ki lorem ipsum ek jargon hai jo aksar design aur printing me use hota hai, jo ek placeholder text hai jo asli content ke jagah par use kiya jata hai.

🤖 Bot: 1350


🤖 Bot: Aapne "good" kaha, shukriya! Maine aapko lorem ipsum ke bare mein jaankari di thi, aur aapko yah jaankari pasand aayi. Agar aapke paas aur koi sawal hai, toh mujhe puchhne mein sankoch na karein!


🤖 Bot: Mera naam nahi hai, main ek smart assistant hoon. Main aapko jaankari aur madad karne ke liye yahaan hoon, lekin mere paas koi vyakti